# Управление экспериментами с MLflow и ClearML

Notebook показывает полный цикл работы с экспериментом:

1. подготовка воспроизводимого split;
2. создание MLflow experiment;
3. несколько запусков с разными параметрами;
4. сравнение запусков и выбор модели по validation;
5. загрузка модели из MLflow и проверка предсказаний;
6. необязательная отправка лучшего результата в ClearML.

Локальная часть MLflow выполняется без отдельного сервера. ClearML по
умолчанию отключён, поскольку для него нужны пользовательские ключи и
доступ к внешнему серверу.


## 1. Импорты и версии

Все обязательные импорты находятся в начале. Если эта ячейка не
выполняется, установите зависимости из `requirements.txt` в отдельное
окружение.


In [1]:
import json
import os
import time
import uuid
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
import sklearn
from mlflow.models import infer_signature
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("MLflow:", mlflow.__version__)
print("scikit-learn:", sklearn.__version__)


MLflow: 3.17.1
scikit-learn: 1.9.1


## 2. Локальное хранилище MLflow

SQLite tracking URI хранит experiment и runs в файле `mlflow.db`.
Такой режим не требует запущенного MLflow server и подходит для
демонстрации на одном компьютере.

После выполнения notebook интерфейс можно открыть командой:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 8080
```


In [2]:
WORK_DIR = Path.cwd()
ARTIFACTS_DIR = (WORK_DIR / "artifacts").resolve()
ARTIFACTS_DIR.mkdir(exist_ok=True)

MLFLOW_DB = (WORK_DIR / "mlflow.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{MLFLOW_DB.as_posix()}")

EXPERIMENT_NAME = "miit-mlops-practice-4-demo"
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

session_id = uuid.uuid4().hex[:12]
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment ID:", experiment_id)
print("Session ID:", session_id)


Tracking URI: sqlite:///D:/Universities/MIIT/MLOps/miit-mlops/practice_4/mlflow.db
Experiment ID: 1
Session ID: e025774da02c


## 3. Данные и split

Используется встроенный Breast Cancer Dataset. Сначала отделяется test,
затем из оставшихся данных выделяется validation. Стратификация
сохраняет долю классов. Test не используется для выбора конфигурации.


In [3]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

X_remain, X_test, y_remain, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_remain,
    y_remain,
    test_size=0.25,
    random_state=42,
    stratify=y_remain,
)

split_summary = pd.DataFrame(
    {
        "rows": [len(X_train), len(X_val), len(X_test)],
        "positive_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
    },
    index=["train", "validation", "test"],
)
split_summary


,rows,positive_rate
train,341,0.627566
validation,114,0.622807
test,114,0.631579


## 4. Функции обучения и оценки

Во всех runs используется один набор метрик. Pipeline включает
стандартизацию и Logistic Regression, поэтому преобразование признаков
сохраняется вместе с моделью.


In [4]:
def build_model(c_value):
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    C=c_value,
                    max_iter=1000,
                    solver="liblinear",
                    random_state=42,
                ),
            ),
        ]
    )


def classification_metrics(model, features, target):
    predictions = model.predict(features)
    probabilities = model.predict_proba(features)[:, 1]
    return {
        "accuracy": accuracy_score(target, predictions),
        "precision": precision_score(target, predictions, zero_division=0),
        "recall": recall_score(target, predictions, zero_division=0),
        "f1": f1_score(target, predictions, zero_division=0),
        "roc_auc": roc_auc_score(target, probabilities),
    }


## 5. Серия запусков

Проверяются три значения `C`. Каждый вариант создаёт отдельный run.
Параметры, validation-метрики, время обучения и модель записываются в
MLflow. Тег `session_id` отделяет текущий запуск notebook от предыдущих.


In [5]:
configurations = [0.1, 1.0, 10.0]
created_run_ids = []

for c_value in configurations:
    run_name = f"logreg-c-{c_value:g}"
    with mlflow.start_run(
        experiment_id=experiment_id,
        run_name=run_name,
        tags={
            "model_family": "logistic_regression",
            "session_id": session_id,
            "dataset": "sklearn_breast_cancer",
        },
    ) as active_run:
        params = {
            "C": c_value,
            "solver": "liblinear",
            "max_iter": 1000,
            "random_state": 42,
        }
        mlflow.log_params(params)

        model = build_model(c_value)
        started_at = time.perf_counter()
        model.fit(X_train, y_train)
        training_seconds = time.perf_counter() - started_at

        validation_metrics = classification_metrics(model, X_val, y_val)
        validation_metrics["training_seconds"] = training_seconds
        mlflow.log_metrics(validation_metrics)

        signature = infer_signature(X_train, model.predict(X_train))
        mlflow.sklearn.log_model(
            sk_model=model,
            name="model",
            signature=signature,
            input_example=X_train.iloc[:5],
        )
        created_run_ids.append(active_run.info.run_id)

print("Created runs:", len(created_run_ids))


Created runs: 3


## 6. Сравнение runs

`search_runs` возвращает параметры, метрики и теги как таблицу. Фильтр
оставляет runs текущего выполнения notebook. Конфигурация выбирается по
validation F1, а не по test.


In [6]:
runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string=f"tags.session_id = '{session_id}'",
    order_by=["metrics.f1 DESC"],
)

comparison_columns = [
    "run_id",
    "tags.mlflow.runName",
    "params.C",
    "metrics.accuracy",
    "metrics.f1",
    "metrics.roc_auc",
    "metrics.training_seconds",
]
comparison = runs[comparison_columns].copy()
comparison


,run_id,tags.mlflow.runName,params.C,metrics.accuracy,metrics.f1,metrics.roc_auc,metrics.training_seconds
0,bbb3bbc5621a42c29da7a4565b23d412,logreg-c-1,1.0,0.982456,0.985915,0.998035,0.003170
1,6d5d751343474d0e941aec9fc0036381,logreg-c-0.1,0.1,0.973684,0.979310,0.997052,0.003183
2,d42e744433ea485883b5e01af257ac00,logreg-c-10,10.0,0.964912,0.971831,0.998035,0.003572


## 7. Загрузка лучшей модели

Лучшая модель загружается по run ID. После загрузки сравниваются
предсказания исходного artifact и повторно загруженной модели. Затем
один раз рассчитываются метрики на test.


In [7]:
best_run_id = comparison.iloc[0]["run_id"]
best_model_uri = f"runs:/{best_run_id}/model"
best_model = mlflow.sklearn.load_model(best_model_uri)

first_predictions = best_model.predict(X_test)
loaded_again = mlflow.sklearn.load_model(best_model_uri)
second_predictions = loaded_again.predict(X_test)
predictions_match = bool((first_predictions == second_predictions).all())

test_metrics = classification_metrics(best_model, X_test, y_test)
with mlflow.start_run(run_id=best_run_id):
    mlflow.log_metrics({f"test_{name}": value for name, value in test_metrics.items()})

verification = {
    "experiment_id": experiment_id,
    "run_id": best_run_id,
    "session_id": session_id,
    "predictions_match": predictions_match,
    "test_metrics": test_metrics,
}
verification_path = ARTIFACTS_DIR / "verification.json"
verification_path.write_text(
    json.dumps(verification, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_artifact(str(verification_path), artifact_path="verification")

print(json.dumps(verification, indent=2, ensure_ascii=False))


{
  "experiment_id": "1",
  "run_id": "bbb3bbc5621a42c29da7a4565b23d412",
  "session_id": "e025774da02c",
  "predictions_match": true,
  "test_metrics": {
    "accuracy": 0.9824561403508771,
    "precision": 0.9861111111111112,
    "recall": 0.9861111111111112,
    "f1": 0.9861111111111112,
    "roc_auc": 0.9953703703703703
  }
}


## 8. Необязательная отправка в ClearML

Перед включением этой ячейки установите ClearML и настройте доступ через
`clearml-init` или переменные окружения. Подставьте свои ключи вместо
`YOUR_CLEARML_ACCESS_KEY` и `YOUR_CLEARML_SECRET_KEY` в локальной
конфигурации, а не в сохраняемом notebook.

Значение `RUN_CLEARML = False` позволяет выполнить notebook целиком без
внешнего сервера. Установите `True`, когда доступ настроен.


In [8]:
RUN_CLEARML = False
CLEARML_PROJECT = "MIIT/MLOps/practice_4/demo"
CLEARML_TASK_NAME = f"best-logreg-{session_id}"

if RUN_CLEARML:
    from clearml import OutputModel, Task

    required_variables = [
        "CLEARML_API_ACCESS_KEY",
        "CLEARML_API_SECRET_KEY",
    ]
    missing_variables = [name for name in required_variables if not os.getenv(name)]
    if missing_variables:
        raise RuntimeError(
            "Не заданы переменные ClearML: " + ", ".join(missing_variables)
        )

    task = Task.init(
        project_name=CLEARML_PROJECT,
        task_name=CLEARML_TASK_NAME,
        reuse_last_task_id=False,
        auto_connect_frameworks=False,
    )
    task.connect(
        {
            "mlflow_experiment_id": experiment_id,
            "mlflow_run_id": best_run_id,
            "session_id": session_id,
        },
        name="provenance",
    )
    logger = task.get_logger()
    for metric_name, metric_value in test_metrics.items():
        logger.report_single_value(f"test_{metric_name}", metric_value)

    comparison_path = ARTIFACTS_DIR / "comparison.csv"
    comparison.to_csv(comparison_path, index=False)
    task.upload_artifact("comparison", artifact_object=str(comparison_path))
    task.upload_artifact("verification", artifact_object=str(verification_path))

    model_path = ARTIFACTS_DIR / "best_model.joblib"
    joblib.dump(best_model, model_path)
    output_model = OutputModel(
        task=task,
        framework="scikit-learn",
        name="breast-cancer-logistic-regression",
    )
    output_model.update_labels({"malignant": 0, "benign": 1})
    output_model.set_metadata(
        key="mlflow_run_id",
        value=best_run_id,
        v_type="str",
    )
    output_model.update_weights(weights_filename=str(model_path))
    output_model.publish()
    print("ClearML Task ID:", task.id)
    print("ClearML Model ID:", output_model.id)
    task.close()
else:
    print("ClearML skipped. Set RUN_CLEARML=True after configuring credentials.")


ClearML skipped. Set RUN_CLEARML=True after configuring credentials.


## 9. Итоговая проверка

Проверка фиксирует минимальный контракт демонстрации. Если assertion
срабатывает, соответствующая часть эксперимента оформлена неправильно.


In [9]:
required_metric_columns = [
    "metrics.accuracy",
    "metrics.f1",
    "metrics.roc_auc",
]

assert len(comparison) == len(configurations)
assert comparison[required_metric_columns].notna().all().all()
assert predictions_match
assert verification_path.exists()

print("Notebook checks passed.")
print("Best run ID:", best_run_id)
print("Test F1:", round(test_metrics["f1"], 4))


Notebook checks passed.
Best run ID: bbb3bbc5621a42c29da7a4565b23d412
Test F1: 0.9861
